In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn import datasets
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_mutual_info_score
import hdbscan
import foscx

In [2]:
quality_measure = "stability"      # change as needed
nearest_neighbors = 30        # change as needed

topM = 5
n_datasets = 20
n_samples = 2000
seed = 30

nk_list = [5, 10, 15, 20, 25]
nd_list = [2, 5, 10, 15, 20, 50, 200]


In [3]:

# Initialize lists to hold multiple datasets
blobs_list = []
aniso_list = []
varied_list = []

for i in range(n_datasets):
    current_seed = seed + i
    for j  in nk_list:
        for k in nd_list:
            
            # Blobs
            blobs_i = datasets.make_blobs(n_samples=n_samples, n_features=k, centers=j, random_state=current_seed)
            blobs_list.append(blobs_i)
            
            # Anisotropicly distributed data
            X, y = datasets.make_blobs(n_samples=n_samples, n_features=k, centers=j, random_state=current_seed)
            transformation = np.random.randn(k, k)
            X_aniso = np.dot(X, transformation)
            aniso_i = (X_aniso, y)
            aniso_list.append(aniso_i)
            
            # Blobs with varied variances
            # Set cluster_std to a list of random values between 0.1 and 5.0 for each cluster
            cluster_std = np.random.uniform(0.1, 5.0, size=j)
            varied_i = datasets.make_blobs(
                n_samples=n_samples, cluster_std=cluster_std, n_features=k, centers=j, random_state=current_seed
            )
            varied_list.append(varied_i)

In [4]:
def get_procedures(k_star):

    return [

        # 1 Baseline
        {
            "name": "Baseline (M=1)",
            "topM": 1,
            "kmin": 2,
            "kmax": None
        },

        # 2 Top-M unconstrained
        {
            "name": "Top 5",
            "topM": 5,
            "kmin": 2,
            "kmax": None
        },

                {
            "name": "Top 10",
            "topM": 10,
            "kmin": 2,
            "kmax": None
        },

        # 3 Fixed cardinality
        {
            "name": "Fixed k = k*",
            "topM": 1,
            "kmin": k_star,
            "kmax": k_star
        },

        # 4 Relaxed cardinality
        {
            "name": "Relaxed k = k* ± 3",
            "topM": 1,
            "kmin": max(2, k_star - 3),
            "kmax": k_star + 3
        },

        # 5 Top-M fixed cardinality
        {
            "name": "Top 5 Fixed k = k*",
            "topM": 5,
            "kmin": k_star,
            "kmax": k_star
        }
    ]

In [6]:
def build_hierarchy(X, method):

    if method == "HDBSCAN*":

        clusterer = hdbscan.HDBSCAN(gen_min_span_tree=True)
        clusterer.fit(X)
        return clusterer

    elif method == "Ward":

        ward = AgglomerativeClustering(
            n_clusters=None,
            distance_threshold=0,
            linkage="ward"
        )

        ward.fit(X)
        return ward

    else:
        raise ValueError("Unknown method")

In [7]:
def run_fosc(X, y, hierarchy, procedure):

    fosc_model = foscx.FOSCX(
        top_M=procedure["topM"],
        kmin=procedure["kmin"],
        kmax=procedure["kmax"],
        quality_measure=quality_measure,
        nearest_neighbors=nearest_neighbors
    )

    fosc_model.fit(hierarchy)

    ari = []

    for c_idx in range(len(fosc_model.candidates_)):
        labels = fosc_model.get_labels(c_idx)
        score = adjusted_mutual_info_score(y, labels)
        ari.append(score)

    if len(ari) == 0:
        return np.nan, np.nan

    ari_first = ari[0]
    ari_best = max(ari)

    return ari_first, ari_best

In [ ]:
def run_benchmark(dataset_list, dataset_name):

    rows = []

    for dataset_id, (X, y) in enumerate(dataset_list):

        k_star = len(np.unique(y))

        procedures = get_procedures(k_star)

        for method in ["Ward", "HDBSCAN*"]:

            hierarchy = build_hierarchy(X, method)

            # ---------------------------------
            # Procedures 1–5
            # ---------------------------------

            for proc in procedures:

                ari_first, ari_best = run_fosc(
                    X, y, hierarchy, proc
                )

                rows.append({
                    "dataset": dataset_name,
                    "dataset_id": dataset_id,
                    "method": method,
                    "procedure": proc["name"],
                    "kmin": proc["kmin"],
                    "kmax": proc["kmax"],
                    "topM": proc["topM"],
                    "ari_first": ari_first,
                    "ari_best": ari_best
                })


    return pd.DataFrame(rows)

In [9]:
results = []

results.append(run_benchmark(blobs_list, "blobs"))
results.append(run_benchmark(aniso_list, "aniso"))
results.append(run_benchmark(varied_list, "varied"))

results_df = pd.concat(results, ignore_index=True)

In [10]:
results_df.to_csv("fosc_synthetic_results.csv", index=False)